# deepMaze — train a heavy agent (DRQN / DTQN) on Colab

Clones the repo, trains the chosen agent, logs everything to MLflow, and writes an `assets/<name>/` bundle ready for the existing pretrained-inference path.

**Required:**
- `MLFLOW_TRACKING_URI` — your deployed MLflow server (see `infra/mlflow/README.md`)
- `GOOGLE_APPLICATION_CREDENTIALS` — only if pushing to `ASSETS_BUCKET` directly. **Upload the JSON via the Colab file pane and reference the path** — do NOT paste the JSON into a cell (it would persist in the .ipynb).

**Outputs:**
- An MLflow run with params, per-episode reward + loss, replay WebP, model weights
- `assets/<run_name>/` zip downloaded to your machine
- Optionally pushed to `gs://${ASSETS_BUCKET}/<run_name>/` for direct backend pickup

## 1 — Config (Colab form fields)

In [ ]:
# @title Config { run: "auto" }
REPO_URL    = "https://github.com/juan-garassino/deepMaze.git"  # @param {type:"string"}
REPO_BRANCH = "main"                                             # @param {type:"string"}

AGENT_TYPE   = "drqn"   # @param ["drqn", "dtqn"]
RUN_NAME     = "drqn_v1" # @param {type:"string"}
MAZE_WIDTH   = 10       # @param {type:"integer"}
MAZE_HEIGHT  = 10       # @param {type:"integer"}
GENERATOR    = "dfs"    # @param ["open", "dfs", "random"]
N_TREASURES  = 1        # @param {type:"integer"}
N_LAVA       = 0        # @param {type:"integer"}
PARTIAL      = 3        # @param {type:"integer"}
NUM_EPISODES = 2000     # @param {type:"integer"}
MAX_STEPS    = 200      # @param {type:"integer"}
EVAL_EPISODES = 20      # @param {type:"integer"}
SEED         = 0        # @param {type:"integer"}

MLFLOW_TRACKING_URI = ""  # @param {type:"string"}
MLFLOW_EXPERIMENT   = "deepmaze"  # @param {type:"string"}
ASSETS_BUCKET       = ""  # @param {type:"string"}

assert MLFLOW_TRACKING_URI, "Set MLFLOW_TRACKING_URI to your tracking server URL"

## 2 — Install repo + deps

In [ ]:
import os
import pathlib
import subprocess
import sys

WORK = pathlib.Path("/content/deepMaze")
if not WORK.exists():
    subprocess.check_call(["git", "clone", "-b", REPO_BRANCH, REPO_URL, str(WORK)])
os.chdir(WORK)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "-r", "requirements.txt",
                       "mlflow==2.18.*", "google-cloud-storage==2.18.*"])
for sub in ("agents", "environment", "training", "utils", "config"):
    p = str(WORK / sub)
    if p not in sys.path:
        sys.path.insert(0, p)

## 3 — Build env + agent

In [ ]:
from maze import MazeEnvironment
from seeding import seed_everything
from train import create_agent, evaluate_agent, simulate_episode, train_agent

seed_everything(SEED)
env = MazeEnvironment(
    width=MAZE_WIDTH, height=MAZE_HEIGHT, density=0.2,
    generator=GENERATOR, n_lava=N_LAVA, n_treasures=N_TREASURES,
    partial_view=PARTIAL, seed=SEED,
)
agent = create_agent(AGENT_TYPE, env)
print(f"env: {MAZE_WIDTH}x{MAZE_HEIGHT}  agent: {type(agent).__name__}")

## 4 — Train with MLflow logging

In [ ]:
import mlflow
from viz_events import EpisodeEvent, EventBus

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT)

params = dict(
    agent_type=AGENT_TYPE, run_name=RUN_NAME,
    maze_width=MAZE_WIDTH, maze_height=MAZE_HEIGHT,
    generator=GENERATOR, n_treasures=N_TREASURES, n_lava=N_LAVA,
    partial=PARTIAL, num_episodes=NUM_EPISODES, max_steps=MAX_STEPS,
    seed=SEED,
)

bus = EventBus()
def _on_ep(ev: EpisodeEvent):
    mlflow.log_metrics(
        {"episode_reward": ev.total_reward,
         "episode_length": ev.length,
         "epsilon": ev.epsilon},
        step=ev.episode,
    )
bus.subscribe(_on_ep)

with mlflow.start_run(run_name=RUN_NAME) as run:
    mlflow.log_params(params)
    train_agent(env, agent, num_episodes=NUM_EPISODES,
                max_steps=MAX_STEPS, bus=bus)
    mean_reward, mean_length, success_rate = evaluate_agent(
        env, agent, num_episodes=EVAL_EPISODES, max_steps=MAX_STEPS,
    )
    mlflow.log_metrics({
        "eval_mean_reward": mean_reward,
        "eval_mean_length": mean_length,
        "eval_success_rate": success_rate,
    })
    RUN_ID = run.info.run_id
print("mlflow run:", RUN_ID, " eval success:", success_rate)

## 5 — Build the `assets/<name>/` bundle

In [ ]:
import json
import shutil
from pathlib import Path

import torch

from maze import RenderMaze
from recorders import ReplayRecorder

OUT = Path(f"assets/{RUN_NAME}")
(OUT / "viz").mkdir(parents=True, exist_ok=True)

config = dict(
    agent_type=AGENT_TYPE,
    net=None,
    maze_width=MAZE_WIDTH, maze_height=MAZE_HEIGHT,
    n_agents=1, density=0.2, generator=GENERATOR,
    no_ensure_solvable=False, n_lava=N_LAVA, lava_reward=-1.0,
    partial=PARTIAL, n_treasures=N_TREASURES, collect_all=False,
    seed=SEED, num_episodes=NUM_EPISODES, max_steps=MAX_STEPS,
    eval_episodes=EVAL_EPISODES, learning_rate=None, discount_factor=0.99,
    image_path=None, sprite_files=["sprites.png"], sprite_size=32,
    replay_fmt="webp", frame_skip=1, max_frames=None,
    policy_snapshot_every=50, live=False, live_web=False, web_port=8000,
    run_id=RUN_ID, run_name=RUN_NAME,
    random_start=False, resume=None, eval_maze="same", eval_seeds=1,
)
(OUT / "config.json").write_text(json.dumps(config, indent=2))

module = getattr(agent, "model", None) or getattr(agent, "ac", None)
torch.save(module.state_dict(), OUT / "model.pt")

agent.set_deterministic(True)
try:
    _, positions, _, total = simulate_episode(env, agent, max_steps=MAX_STEPS, at_start=True)
finally:
    agent.set_deterministic(False)

full_frames = [env.maze.copy() for _ in positions]
sprites = RenderMaze.placeholder_sprites(32)
rm = RenderMaze(sprites)
ReplayRecorder(rm).feed(full_frames, positions, None)
rm.save(str(OUT / "viz" / "replay.webp"), fmt="webp")

print("bundle:", sorted(p.relative_to(OUT).as_posix() for p in OUT.rglob("*") if p.is_file()))

## 6 — Log bundle to MLflow + optionally push to GCS

In [ ]:
with mlflow.start_run(run_id=RUN_ID):
    mlflow.log_artifacts(str(OUT), artifact_path=f"assets/{RUN_NAME}")

if ASSETS_BUCKET:
    from google.cloud import storage
    bucket = storage.Client().bucket(ASSETS_BUCKET)
    for f in OUT.rglob("*"):
        if f.is_file():
            blob = bucket.blob(f"{RUN_NAME}/{f.relative_to(OUT)}")
            blob.upload_from_filename(str(f))
            print("→ gs://" + ASSETS_BUCKET + "/" + blob.name)
    print("bundle pushed; trigger a backend redeploy to pick it up")
else:
    shutil.make_archive(f"/content/{RUN_NAME}", "zip", "assets", RUN_NAME)
    print(f"/content/{RUN_NAME}.zip ready — download via the Colab file pane")

## Next steps

- Unzip into `assets/<name>/` locally and run `python web/server.py` → pretrained dropdown.
- Or call the `promote_flow` Prefect flow (D) with `RUN_ID` to commit the bundle and redeploy automatically.
- See MLflow run: `${MLFLOW_TRACKING_URI}/#/experiments/.../runs/{RUN_ID}`.